In [1]:
import os
import shutil
from pathlib import Path
from tqdm import tqdm

def oversample_rare_classes(yolo_dataset_dir: str, rare_class_idx: int = 5, multiplier: int = 10):
    """
    Дублирует текстовые лейблы и изображения, содержащие редкий класс,
    чтобы модель видела их чаще во время эпохи.
    """
    train_labels_dir = Path(yolo_dataset_dir) / "labels" / "train"
    train_images_dir = Path(yolo_dataset_dir) / "images" / "train"
    
    label_files = list(train_labels_dir.glob("*.txt"))
    print(f"Сканирование разметки на наличие редкого класса (ID: {rare_class_idx})...")
    
    copied_count = 0
    for label_path in tqdm(label_files):
        with open(label_path, "r") as f:
            lines = f.readlines()
        
        # Проверяем, есть ли на этой картинке целевой редкий класс
        has_rare_class = any(line.strip().startswith(f"{rare_class_idx} ") for line in lines)
        
        if has_rare_class:
            img_ext = ".jpg" # или .png в зависимости от того, что сохранил скрипт
            img_path = train_images_dir / f"{label_path.stem}{img_ext}"
            
            if not img_path.exists():
                img_path = train_images_dir / f"{label_path.stem}.png"
            
            if img_path.exists():
                # Создаем несколько копий файла с уникальными именами
                for i in range(1, multiplier):
                    new_base_name = f"{label_path.stem}_os_{i}"
                    
                    # Копируем лейбл
                    shutil.copy2(label_path, train_labels_dir / f"{new_base_name}.txt")
                    # Копируем картинку
                    shutil.copy2(img_path, train_images_dir / f"{new_base_name}{img_path.suffix}")
                    
                copied_count += 1

    print(f"Апсемплинг завершен. {copied_count} уникальных изображений размножено в {multiplier} раз(а).")

oversample_rare_classes(r'D:\Developments of Cognify\datasets\Agriculture-Vision-2021\yolo_dataset\rgb', rare_class_idx=5, multiplier=12)

Сканирование разметки на наличие редкого класса (ID: 5)...


100%|██████████| 56944/56944 [07:45<00:00, 122.23it/s]

Апсемплинг завершен. 356 уникальных изображений размножено в 12 раз(а).


In [ ]:
import tarfile
from pathlib import Path
from tqdm import tqdm  

def archive_dataset_jupyter(source_dir: Path, output_tar: Path):
    """
    Высокоскоростная упаковка датасета в TAR-архив без сжатия.
    """
    src = Path(source_dir).resolve()
    out = Path(output_tar).resolve()
    
    if not src.exists():
        print(f"❌ Ошибка: Папка {src} не существует! Проверьте правильность пути.")
        return
        
    out.parent.mkdir(parents=True, exist_ok=True)
    
    print(f"🔍 Сканирование файлов в папке: {src}")
    all_files = [p for p in src.rglob('*') if p.is_file()]
    print(f"📦 Найдено файлов для упаковки: {len(all_files)}")
    
    if len(all_files) == 0:
        print("⚠️ Внимание: Папка пуста! Архивация отменена.")
        return
    
    print("⚡ Запуск архивации...")
    with tarfile.open(out, "w") as tar:
        for file_path in tqdm(all_files, desc="Сборка TAR-архива"):
            arcname = file_path.relative_to(src.parent)
            tar.add(file_path, arcname=arcname)
            
    print(f"🏁 Готово! Архив успешно сохранен: {out}")
    print(f"📊 Итоговый размер архива: {out.stat().st_size / (1024 * 1024 * 1024):.2f} GB")

DATASET_PATH = Path(r"D:\Developments of Cognify\datasets\Agriculture-Vision-2021\yolo_dataset")

ARCHIVE_PATH = DATASET_PATH.parent / "results" / "yolo_dataset_final.tar"

archive_dataset_jupyter(DATASET_PATH, ARCHIVE_PATH)

🔍 Сканирование файлов в папке: D:\Developments of Cognify\datasets\Agriculture-Vision-2021\yolo_dataset
📦 Найдено файлов для упаковки: 348360
⚡ Запуск архивации...


Сборка TAR-архива: 100%|██████████| 348360/348360 [48:50<00:00, 118.87it/s]


🏁 Готово! Архив успешно сохранен: D:\Developments of Cognify\datasets\Agriculture-Vision-2021\results\yolo_dataset_final.tar
📊 Итоговый размер архива: 8.95 GB
